### CUSTOMERS

In [0]:
df_cust = spark.read.table("pysparkdbt.bronze.customers")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import *
import os
import sys
from typing import List
from pyspark.sql import DataFrame
from pyspark.sql.window import Window
from delta.tables import DeltaTable


In [0]:
df_cust = df_cust.withColumn("domain", split("email", "@"). getItem(1))

df_cust = df_cust.withColumn("phone_number", regexp_replace(col("phone_number"), r"[^0-9]", ""))

df_cust = df_cust.withColumn("full_name", concat_ws(" ", col("first_name"), col("last_name")))

df_cust = df_cust.drop("first_name", "last_name")



##### FUNCTIONS



In [0]:
class transformations:
    def dedup(self, df:DataFrame, dedup_cols:List, cdc:str):


        df = df.withColumn("dedupKey", concat(*dedup_cols))

        df = df.withColumn("dedupCounts", row_number().over(Window.partitionBy("dedupKey").orderBy(desc(cdc))))

        df = df.filter(col("dedupCounts")==1)

        df = df.drop("dedupKey", "dedupCounts")
        
        return df
    
    def process_timestamp(self, df):

        df = df.withColumn("process_timestamp", current_timestamp())

        return df
    
    def upsert(self, df, key_cols, table, cdc):

        merge_condition = " AND ".join([f"s.{i} = t.{i}" for i in key_cols])

        dlt_object = DeltaTable.forName(spark, f"pysparkdbt.silver.{table}")

        dlt_object.alias("t").merge(df.alias("s"), merge_condition)\
            .whenMatchedUpdateAll(condition=f"s.{cdc} >= t.{cdc}")\
            .whenNotMatchedInsertAll()\
            .execute() 

        return 1

In [0]:
cust_obj = transformations()

df_cust = cust_obj.dedup(df_cust, ["customer_id"], "last_updated_timestamp")

df_cust = cust_obj.process_timestamp(df_cust)





In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.customers"):

    df_cust.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.customers")
else:

    cust_obj.upsert(df_cust, ["customer_id"], "customers", "last_updated_timestamp")


### DRIVERS


In [0]:
df_drivers = spark.read.table("pysparkdbt.bronze.drivers")

In [0]:
df_drivers = df_drivers.withColumn("phone_number", regexp_replace(col("phone_number"), r"[^0-9]", ""))

df_drivers = df_drivers.withColumn("full_name", concat_ws(" ", col("first_name"), col("last_name")))

df_drivers = df_drivers.drop("first_name", "last_name")




In [0]:
driver_obj = transformations()

df_drivers = driver_obj.dedup(df_drivers, ["driver_id"], "last_updated_timestamp")

df_drivers = driver_obj.process_timestamp(df_drivers)


In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.drivers"):

    df_drivers.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.drivers")
else:

    cust_obj.upsert(df_drivers, ["driver_id"], "drivers", "last_updated_timestamp")


### LOCATIONS

In [0]:
df_locations = spark.read.table("pysparkdbt.bronze.locations")

In [0]:
loc_objt = transformations()

df_locations = loc_objt.dedup(df_locations, ["location_id"], "last_updated_timestamp")

df_locations = loc_objt.process_timestamp(df_locations)

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.locations"):

    df_locations.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.locations")

else:

    loc_objt.upsert(df_locations, ["location_id"], "locations", "last_updated_timestamp")

### PAYMENTS

In [0]:
df_pays = spark.read.table("pysparkdbt.bronze.payments")

In [0]:
df_pays = df_pays.withColumn("online_payment_status",
            when(((col("payment_method")=="Card") & (col("payment_status")=="Success")), "online-success")
            .when(((col("payment_method")=="Card") & (col("payment_status")=="Failed")), "online-failed")
            .when(((col("payment_method")=="Card") & (col("payment_status")=="Pending")), "online-pending")
            .otherwise("offline")
)

df_pays.display()

In [0]:
pay_objt = transformations()

df_pays = pay_objt.dedup(df_pays, ["payment_id"], "last_updated_timestamp")

df_pays = pay_objt.process_timestamp(df_pays)

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.payments"):

    df_pays.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.payments")

else:
    
    pay_objt.upsert(df_pays, ["payment_id"], "payments", "last_updated_timestamp")


### TRIPS

In [0]:
df_trips = spark.read.table("pysparkdbt.bronze.trips")

In [0]:
trips_objt = transformations()

df_trips = trips_objt.dedup(df_trips, ["trip_id"], "last_updated_timestamp")

df_trips = trips_objt.process_timestamp(df_trips)


In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.trips"):

    df_trips.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.trips")

else:

    trips_objt.upsert(df_trips, ["trip_id"], "trips", "last_updated_timestamp")

In [0]:
%sql
drop table pysparkdbt.silver.trips

### VEHICLES

In [0]:
df_vehicles = spark.read.table("pysparkdbt.bronze.vehicles")

In [0]:
df_vehicles = df_vehicles.withColumn("make", upper(col("make")))

In [0]:
vehicles_objt = transformations()

df_vehicles = vehicles_objt.dedup(df_vehicles, ["vehicle_id"], "last_updated_timestamp")

df_vehicles = vehicles_objt.process_timestamp(df_vehicles)


In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.vehicles"):

    df_vehicles.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.vehicles")

else:

    vehicles_objt.upsert(df_vehicles, ["vehicle_id"], "vehicles", "last_updated_timestamp")
